After any interrupted run, restart the runtime before re-running this notebook. Re-running %pip install in a live session produces exactly this stale-import failure.

# Phase 6 v2: frozen benchmark evaluation

This evaluation-only notebook compares the pinned base model and the saved Phase 6 adapter on the frozen 60-case benchmark. It does not train, alter, or expose benchmark labels to either model. Reports are written to private Google Drive.

In [ ]:
%pip install -q transformers==5.14.1 accelerate==1.14.0
%pip install -q bitsandbytes==0.50.0 peft==0.20.0 safetensors==0.8.0

import subprocess
import sys
from pathlib import Path

RUN_DIR_NAME = None

REPOSITORY_URL = "https://github.com/muzzary/GTM-Agent.git"
REPOSITORY_REF = "codex/phase-6-reviewed-adapter"
REPOSITORY_DIR = Path("/content/GTM-Agent")
if REPOSITORY_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "fetch", "origin", REPOSITORY_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "checkout", REPOSITORY_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPOSITORY_REF,
            REPOSITORY_URL,
            str(REPOSITORY_DIR),
        ],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPOSITORY_DIR)],
    check=True,
)
sys.path.insert(0, str(REPOSITORY_DIR))

In [ ]:
import importlib
import importlib.metadata

PINNED_VERSIONS = {
    "transformers": "5.14.1",
    "accelerate": "1.14.0",
    "bitsandbytes": "0.50.0",
    "peft": "0.20.0",
    "safetensors": "0.8.0",
}
RESTART_INSTRUCTION = (
    "Runtime > Restart session, then run all cells top to bottom. "
    "Do not re-run the install cell in a live session."
)
packages_without_version = []
for package_name, pinned_version in PINNED_VERSIONS.items():
    installed_version = importlib.metadata.version(package_name)
    imported_version = getattr(
        importlib.import_module(package_name), "__version__", None
    )
    if installed_version != pinned_version or (
        imported_version is not None and imported_version != installed_version
    ):
        raise RuntimeError(
            f"{package_name} version mismatch: installed={installed_version}, "
            f"imported={imported_version}. {RESTART_INSTRUCTION}"
        )
    if imported_version is None:
        packages_without_version.append(package_name)
print({"packages_without_version": packages_without_version})

In [ ]:
import gc
import json
import re
import time
from datetime import UTC, datetime

import torch
from google.colab import drive
from peft import PeftModel
from pydantic import ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from src.evaluation.phase1 import load_manifest as load_phase1_manifest
from src.evaluation.phase6_benchmark import (
    audit_phase6_benchmark,
    load_phase6_benchmark,
)
from src.evaluation.phase6_v2 import (
    compare_phase6_v2_reports,
    run_phase6_v2_evaluation,
)
from src.schemas.dataset import DatasetManifest, DatasetManifestV2
from src.schemas.inference import GroundedOutreachOutput, ModelIdentity
from src.schemas.training import AdapterArtifactMetadata, TrainingConfigV2

CONFIG_PATH = REPOSITORY_DIR / "configs/phase6/training-v2.json"
PILOT_PATH = REPOSITORY_DIR / "configs/phase6/pilot.json"
PHASE1_PATH = REPOSITORY_DIR / "configs/phase1/benchmark.json"
V2_DATASET_PATH = REPOSITORY_DIR / "configs/phase6/dataset-v2.json"
V2_BENCHMARK_PATH = REPOSITORY_DIR / "configs/phase6/benchmark-v2.json"

config = TrainingConfigV2.model_validate_json(CONFIG_PATH.read_text(encoding="utf-8"))
pilot = DatasetManifest.model_validate_json(PILOT_PATH.read_text(encoding="utf-8"))
v2_dataset = DatasetManifestV2.model_validate_json(
    V2_DATASET_PATH.read_text(encoding="utf-8")
)
phase1 = load_phase1_manifest(PHASE1_PATH)
v2_benchmark = load_phase6_benchmark(V2_BENCHMARK_PATH)
assert config.base_model_id == "Qwen/Qwen3-4B-Instruct-2507"
assert config.base_model_revision == "cdbee75f17c01a7cc42f958dc650907174af0554"
blocked_identities = {
    *(f"product:{item.product_group}" for item in pilot.examples),
    *(f"icp:{item.icp_group}" for item in pilot.examples),
    *(f"company:{item.company_group}" for item in pilot.examples),
    *(f"prospect:{item.prospect_group}" for item in pilot.examples),
}
benchmark_audit = audit_phase6_benchmark(
    v2_benchmark,
    blocked_identity_groups=blocked_identities,
    blocked_case_ids={case.case_id for case in phase1.cases},
)
assert benchmark_audit.evaluation_ready
assert benchmark_audit.total_cases == 60
assert torch.cuda.is_available(), "Select a Colab GPU runtime before evaluation."

drive.mount("/content/drive")
ARTIFACT_ROOT = Path("/content/drive/MyDrive/gtm-agent-phase6")
run_pattern = re.compile(r"^run-(\d{2,})$")
def has_completed_adapter(path):
    return (path / "adapter-metadata.json").is_file()

def resolve_adapter_directory(run_dir):
    adapter_dir = run_dir / "adapter"
    if has_completed_adapter(adapter_dir):
        return adapter_dir
    checkpoints_dir = run_dir / "checkpoints"
    checkpoint_dirs = sorted(
        (
            path
            for path in checkpoints_dir.glob("epoch-*")
            if path.is_dir() and has_completed_adapter(path)
        ),
        key=lambda path: int(path.name.split("-")[-1]),
        reverse=True,
    )
    return checkpoint_dirs[0] if checkpoint_dirs else None

searched_run_names = sorted(
    (
        path.name
        for path in ARTIFACT_ROOT.iterdir()
        if path.is_dir() and run_pattern.fullmatch(path.name)
    ),
    key=lambda name: int(name.split("-")[-1]),
    reverse=True,
)
if RUN_DIR_NAME is None:
    run_dirs = [ARTIFACT_ROOT / name for name in searched_run_names]
    RUN_DIR = next(
        (path for path in run_dirs if resolve_adapter_directory(path)),
        None,
    )
else:
    RUN_DIR = ARTIFACT_ROOT / RUN_DIR_NAME
if RUN_DIR is None:
    raise RuntimeError(
        "No run directory contains a completed adapter; searched: "
        + (", ".join(searched_run_names) or "none")
    )
print({"resolved_run_directory": str(RUN_DIR)})
ADAPTER_DIR = resolve_adapter_directory(RUN_DIR)
if ADAPTER_DIR is None:
    raise RuntimeError(
        f"Run {RUN_DIR.name} has no completed adapter or checkpoint."
    )
METADATA_PATH = ADAPTER_DIR / "adapter-metadata.json"
metadata = AdapterArtifactMetadata.model_validate_json(
    METADATA_PATH.read_text(encoding="utf-8")
)
assert metadata.adapter_id == config.adapter_id, (
    f"Adapter ID mismatch: expected {config.adapter_id}, found {metadata.adapter_id}."
)
assert metadata.dataset_id == v2_dataset.dataset_id, (
    f"Dataset ID mismatch: expected {v2_dataset.dataset_id}, "
    f"found {metadata.dataset_id}."
)
assert metadata.dataset_version == v2_dataset.dataset_version, (
    f"Dataset version mismatch: expected {v2_dataset.dataset_version}, "
    f"found {metadata.dataset_version}."
)
assert metadata.base_model_id == config.base_model_id
assert metadata.base_model_revision == config.base_model_revision
print(
    {
        "resolved_adapter_directory": str(ADAPTER_DIR),
        "epochs_completed": metadata.epochs_completed,
    }
)
print(
    {
        "benchmark": v2_benchmark.benchmark_id,
        "benchmark_sha256": v2_benchmark.content_sha256,
        "cases": benchmark_audit.total_cases,
        "adapter_id": metadata.adapter_id,
        "adapter_revision": metadata.adapter_revision,
    }
)

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
BASE_IDENTITY = ModelIdentity(
    model_id=config.base_model_id,
    model_revision=config.base_model_revision,
)
ADAPTER_IDENTITY = ModelIdentity(
    model_id=config.base_model_id,
    model_revision=config.base_model_revision,
    adapter_id=metadata.adapter_id,
    adapter_revision=metadata.adapter_revision,
)

class ModelOutputError(ValueError):
    def __init__(self, message, raw_output):
        super().__init__(message)
        self.raw_output_excerpt = raw_output[:2000]


def load_evaluation_model(adapter_dir=None):
    tokenizer = AutoTokenizer.from_pretrained(
        config.base_model_id,
        revision=config.base_model_revision,
        trust_remote_code=False,
        use_fast=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        config.base_model_id,
        revision=config.base_model_revision,
        quantization_config=quantization,
        device_map="auto",
        trust_remote_code=False,
        use_safetensors=True,
    )
    if adapter_dir is not None:
        model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
    model.eval()
    return tokenizer, model


def generate_grounded_output(request, tokenizer, model):
    messages = [
        {"role": "system", "content": "Return strict JSON only."},
        {"role": "user", "content": request.prompt},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    torch.manual_seed(request.seed)
    torch.cuda.manual_seed_all(request.seed)
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **encoded,
            do_sample=False,
            max_new_tokens=request.max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0, encoded["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(generated, skip_special_tokens=True).strip()
    try:
        parsed = json.loads(raw_output)
        result = GroundedOutreachOutput.model_validate(parsed)
    except (json.JSONDecodeError, ValidationError) as error:
        raise ModelOutputError(
            f"model output does not match the v2 contract: {error}",
            raw_output,
        ) from error
    print(
        {
            "request_id": request.request_id,
            "latency_seconds": round(time.perf_counter() - started, 2),
            "status": result.generation_status,
        }
    )
    return result


def evaluate_model(tokenizer, model, identity):
    return run_phase6_v2_evaluation(
        v2_benchmark,
        identity,
        lambda request: generate_grounded_output(request, tokenizer, model),
        max_new_tokens=768,
        seed=42,
    )


RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
EVALUATION_DIR = RUN_DIR / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
print({"evaluation_directory": str(EVALUATION_DIR)})

In [ ]:
base_tokenizer, base_model = load_evaluation_model()
base_report = evaluate_model(base_tokenizer, base_model, BASE_IDENTITY)
BASE_REPORT_PATH = EVALUATION_DIR / "phase6-v2-base-report.json"
BASE_REPORT_PATH.write_text(
    base_report.model_dump_json(indent=2),
    encoding="utf-8",
)
del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(
    {
        "base_report": str(BASE_REPORT_PATH),
        "valid_outputs": base_report.valid_output_count,
        "deterministic_passes": base_report.deterministic_passed_case_count,
    }
)

In [ ]:
adapter_tokenizer, adapter_model = load_evaluation_model(ADAPTER_DIR)
adapter_report = evaluate_model(
    adapter_tokenizer, adapter_model, ADAPTER_IDENTITY
)
ADAPTER_REPORT_PATH = EVALUATION_DIR / "phase6-v2-adapter-report.json"
ADAPTER_REPORT_PATH.write_text(
    adapter_report.model_dump_json(indent=2),
    encoding="utf-8",
)
comparison = compare_phase6_v2_reports(base_report, adapter_report)
COMPARISON_PATH = EVALUATION_DIR / "phase6-v2-comparison.json"
COMPARISON_PATH.write_text(
    comparison.model_dump_json(indent=2),
    encoding="utf-8",
)
print(comparison.model_dump(mode="json"))
print(
    {
        "evaluation_directory": str(EVALUATION_DIR),
        "base_report": str(BASE_REPORT_PATH),
        "adapter_report": str(ADAPTER_REPORT_PATH),
        "comparison": str(COMPARISON_PATH),
    }
)

## Reading the result

`inconclusive` means the adapter missed the 95% valid-output threshold or at least one deterministic case gate. `pending_semantic_review` means the outputs are structurally ready for blind human scoring; it is not acceptance. The Phase 6 quality gate remains unaccepted until the semantic rubric is completed.